# 03 — Receptor preparation for docking (AChE)

Run once in Google Colab. Produces `receptor.pdbqt` and `box_config.json`,
which the Streamlit app's **Docking** tab needs. Commit both files to your
GitHub repo alongside `app.py` so docking works out of the box for every user
— no local install needed.

Default structure: PDB `4EY7` (human acetylcholinesterase with donepezil
bound). To dock against a different Alzheimer's target (BACE1, etc.), swap
`PDB_ID` for that target's PDB entry.


In [ ]:
!pip install -q requests pdbfixer openmm meeko rdkit


In [ ]:
import json
import requests
import numpy as np

PDB_ID = "4EY7"

r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb")
r.raise_for_status()
with open(f"{PDB_ID}.pdb", "w") as f:
    f.write(r.text)
print(f"Downloaded {PDB_ID}.pdb")


### Find the docking box

Before stripping the co-crystallized ligand (needed to make a clean receptor),
use its coordinates to define where the docking box should sit — the pocket
a real inhibitor already occupies is the right place to search.

In [ ]:
def get_ligand_centroid(pdb_path, exclude_resnames=("HOH", "WAT")):
    coords_by_res = {}
    with open(pdb_path) as f:
        for line in f:
            if line.startswith("HETATM"):
                resname = line[17:20].strip()
                if resname in exclude_resnames:
                    continue
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                coords_by_res.setdefault(resname, []).append((x, y, z))
    if not coords_by_res:
        return None, None
    # Assume the largest heteroatom group is the bound ligand (water is excluded above)
    largest_res = max(coords_by_res, key=lambda k: len(coords_by_res[k]))
    coords = np.array(coords_by_res[largest_res])
    return coords.mean(axis=0), largest_res

centroid, ligand_resname = get_ligand_centroid(f"{PDB_ID}.pdb")
print("Detected bound ligand:", ligand_resname)
print("Pocket centroid (x, y, z):", centroid)


In [ ]:
box_config = {
    "center_x": float(centroid[0]),
    "center_y": float(centroid[1]),
    "center_z": float(centroid[2]),
    "size_x": 20.0,
    "size_y": 20.0,
    "size_z": 20.0,
}
with open("box_config.json", "w") as f:
    json.dump(box_config, f, indent=2)
box_config


### Clean the receptor and convert to PDBQT

Removes waters/ligand, fills in missing atoms, adds hydrogens at physiological
pH, then converts to the `.pdbqt` format Vina requires.

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

fixer = PDBFixer(filename=f"{PDB_ID}.pdb")
fixer.removeHeterogens(keepWater=False)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

with open("receptor_clean.pdb", "w") as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)
print("Wrote receptor_clean.pdb")


In [ ]:
# meeko's receptor-prep CLI converts the cleaned PDB to PDBQT.
# Flags occasionally change between meeko versions -- if this errors, run
# `!mk_prepare_receptor.py --help` in a new cell to see the current options
# for your installed version and adjust accordingly.
!mk_prepare_receptor.py --pdb receptor_clean.pdb -o receptor -p


In [ ]:
import os
assert os.path.exists("receptor.pdbqt"), "receptor.pdbqt was not created -- check the mk_prepare_receptor.py output above"
print("receptor.pdbqt ready:", os.path.getsize("receptor.pdbqt"), "bytes")

from google.colab import files
files.download("receptor.pdbqt")
files.download("box_config.json")


**Next step:** put `receptor.pdbqt` and `box_config.json` in the same GitHub folder as `app.py`. The Docking tab will pick them up automatically; anyone using the app can still upload their own instead.